In [ ]:
#import cartopy.crs as ccrs
import numpy as np
#%matplotlib nbagg
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import ICRS    
from astropy.coordinates import Galactic 
from astropy.coordinates import SkyCoord 
from mpl_toolkits.axes_grid1 import make_axes_locatable
from reproject import reproject_interp
from reproject import reproject_from_healpix
import healpy as hp
from matplotlib import cm
from reproject import reproject_to_healpix
from astropy_healpix import HEALPix
from astropy.io import fits
from importlib import reload
import astropy.units as u
from astropy.convolution import Gaussian2DKernel
#from skyfield.api import Star, load, wgs84
#from skyfield.data import hipparcos
#from skyfield.projections import build_stereographic_projection
import matplotlib.path as mpath
from astropy.convolution import convolve
from matplotlib.patches import Ellipse

## Constellations from Steven's annotation files:

In [ ]:
with open('/srv/data/constellations/const_all.ann') as f:
    const_all = [line.rstrip() for line in f]

const_lon_list_all = []
const_lat_list_all = []
const_mag_list_all = []
    
for i in range(0,len(const_all)):
    
    const_file = const_all[i][23:36]
    print(i,const_file)
    
    const_lon_list = []
    const_lat_list = []
    const_mag_list = []
    
    with open('/srv/data/constellations/'+const_file) as f:
        const = [line.rstrip() for line in f]
    for j in range(1,len(const)):
        if const[j]:
            if ((const[j].split()[0] != 'CLINES') & (const[j].split()[0] != '#') & (const[j].split()[0] != 'DLINES')):
                print(const[j].split()[0],const[j].split()[1],const[j].split()[3])
                const_lon_list.append(const[j].split()[0])
                const_lat_list.append(const[j].split()[1])
                const_mag_list.append(const[j].split()[3])
                
            const_lon_arr = np.array(const_lon_list).astype(float)
            const_lat_arr = np.array(const_lat_list).astype(float)
            const_mag_arr = np.array(const_mag_list).astype(float)
            
            idx_keep = np.where(const_lon_arr > 0)
            
            const_lon_arr = const_lon_arr[idx_keep]
            const_lat_arr = const_lat_arr[idx_keep]
            const_mag_arr = const_mag_arr[idx_keep]
            
    const_lon_list_all.append(const_lon_arr)
    const_lat_list_all.append(const_lat_arr)
    const_mag_list_all.append(const_mag_arr)
    print('')

In [ ]:
for i in range(0,88):
    print(const_lon_list_all[i])
    print(const_lat_list_all[i])
    print(const_mag_list_all[i])
    print('')

In [ ]:
fig,ax = plt.subplots(1,figsize=(20,10))
ax.grid()

for i in range(0,88):
    #marker_size = 100 * 10 ** (const_mag_list_all[i] / -2.5)
    #ax.scatter(const_lon_list_all[i],const_lat_list_all[i],s=marker_size)
    ax.scatter(const_lon_list_all[i],const_lat_list_all[i],s=10)
ax.set_xlim(360,0)
ax.set_ylim(-90,90)

In [ ]:
const_lon_all = np.concatenate(const_lon_list_all)
const_lat_all = np.concatenate(const_lat_list_all)
const_mag_all = np.concatenate(const_mag_list_all)
print(len(const_lon_all))

plt.hist(np.array(const_mag_all),bins=20);


In [ ]:
print(np.sqrt(2))
print(np.sqrt(49))
print(np.sqrt(121))

In [ ]:
marker_size = np.empty_like(const_mag_all)
marker_clr  = np.empty_like(const_mag_all)
print(marker_size.shape)

marker_size[ const_mag_all >= 5] = 2
marker_size[(const_mag_all >= 4) & (const_mag_all < 5)] = 2**2
marker_size[(const_mag_all >= 3) & (const_mag_all < 4)] = 3**2
marker_size[(const_mag_all >= 2) & (const_mag_all < 3)] = 4**2
marker_size[(const_mag_all >= 1) & (const_mag_all < 2)] = 6**2
marker_size[ const_mag_all < 1] = 10**2

#marker_size = 100 * 10 ** (const_mag_all / -2.5)

marker_clr[ const_mag_all >= 5] = 0
marker_clr[(const_mag_all >= 4) & (const_mag_all < 5)] = 1
marker_clr[(const_mag_all >= 3) & (const_mag_all < 4)] = 2
marker_clr[(const_mag_all >= 2) & (const_mag_all < 3)] = 3
marker_clr[(const_mag_all >= 1) & (const_mag_all < 2)] = 4
marker_clr[ const_mag_all < 1] = 5
marker_clr = ['C'+x for x in np.array(marker_clr).astype(int).astype(str)]

#for i in range(0,len(marker_size)):
#    print(const_mag_all[i],marker_size[i])

fig,ax = plt.subplots(1,figsize=(20,10))
ax.grid()
ax.scatter(const_lon_all,const_lat_all,s=marker_size,color=marker_clr)
ax.set_xlim(360,0)

In [ ]:
coords = SkyCoord(l=const_lon_all*u.deg, b=const_lat_all*u.deg,frame=Galactic)

ra_constel  = coords.icrs.ra.deg
dec_constel = coords.icrs.dec.deg

fig,ax = plt.subplots(1,figsize=(20,10))

ax.scatter(ra_constel,dec_constel,s=marker_size,color=marker_clr)

ax.grid()
ax.set_xlim(360,0)

In [ ]:
from matplotlib.patches import Ellipse

fig,ax = plt.subplots(1,figsize=(20,10))

ells = [Ellipse(xy = (ra_constel[i],dec_constel[i]), 
                width  = 0.3*np.sqrt(marker_size[i])/(np.cos(dec_constel[i]*np.pi/180)), 
                height = 0.3*np.sqrt(marker_size[i]))
        for i in range(0,len(ra_constel))]

ax.grid()
ax.set_xlim(360,0)
ax.set_ylim(-90,90)

for e in ells:
    ax.add_artist(e)
    e.set_clip_box(ax.bbox)
    e.set_facecolor('k')


## Becky's beam convolving function:

In [ ]:
def convolve_to_beam(mapin, biggest_beam, current_beam,hdr, output = False):
    
    fwhm = np.sqrt(biggest_beam**2 - current_beam**2) * 60

    if output: print('Convolving', np.round(current_beam,2), 'deg to', np.round(biggest_beam,2), 'deg\t FWHM of kernel: '+str(fwhm/60)+' degrees')   
    pixsize = hdr['CDELT2']*60
    
    if fwhm < pixsize:
        if output: print("Just returning the map without change")
        return mapin
    
    if output: print('pixel size: '+str(pixsize)+' arcmin')   
    stddev = np.round((fwhm/2.355)/pixsize,0)
    if output: print('standard deviation of kernel: '+str(stddev)+' pixels')
    
    if stddev == 0:
        return mapin

    kernel = Gaussian2DKernel(x_stddev=stddev,y_stddev=stddev)
    
    return convolve(mapin,kernel)

## Read in GMIMS DRAO (galactic)

In [ ]:
hdu_drao = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_FD_PI.fits')
hdr_drao = hdu_drao[0].header
data_drao = hdu_drao[0].data
print(data_drao.shape)

FD_drao = WCS(hdr_drao).all_pix2world(0,0,range(data_drao.shape[0]),0)[2]
print(FD_drao[0],FD_drao[-1])

## Read in STAPS (equatorial)

In [ ]:
hdu_staps = fits.open('/srv/data/staps/Jan2024/P.cln.fits')
hdr_staps = hdu_staps[0].header
data_staps = hdu_staps[0].data
print(data_staps.shape)

FD_staps = WCS(hdr_staps).all_pix2world(0,0,range(data_staps.shape[0]),0)[2]
print(FD_staps[0],FD_staps[-1])
print(repr(hdr_staps))

# Make new 2D equatorial and galactic headers

In [ ]:
hdu = fits.PrimaryHDU()
hdr_eq = hdu.header
del hdr_eq['EXTEND']
hdr_eq['BITPIX']   = -32
hdr_eq['NAXIS']    = 2
hdr_eq['NAXIS1']   = 5760#2880
hdr_eq['NAXIS2']   = 2880#1440
hdr_eq['CUNIT1']   = 'deg     '
hdr_eq['CUNIT2']   = 'deg     '
hdr_eq['COORDSYS'] = 'icrs    '
hdr_eq['CTYPE1']   = 'RA---CAR'
hdr_eq['CRPIX1']   = 2880.5#1440.5
hdr_eq['CRVAL1']   = 180.
hdr_eq['CDELT1']   = -0.0625#-0.125
hdr_eq['CTYPE2']   = 'DEC--CAR'
hdr_eq['CRPIX2']   = 1440.5#720.5 
hdr_eq['CRVAL2']   = 0.
hdr_eq['CDELT2']   = 0.0625#0.125
#print(repr(hdr_eq))

In [ ]:
hdu = fits.PrimaryHDU()
hdr_gal = hdu.header
del hdr_gal['EXTEND']
hdr_gal['BITPIX']   = -32
hdr_gal['NAXIS']    = 2
hdr_gal['NAXIS1']   = 5760#2880
hdr_gal['NAXIS2']   = 2880#1440
hdr_gal['CUNIT1']   = 'deg     '
hdr_gal['CUNIT2']   = 'deg     '
hdr_gal['COORDSYS'] = 'icrs    '
hdr_gal['CTYPE1']   = 'GLON-CAR'
hdr_gal['CRPIX1']   = 2880.5#1440.5
hdr_gal['CRVAL1']   = 180.
hdr_gal['CDELT1']   = -0.0625#-0.125
hdr_gal['CTYPE2']   = 'GLAT-CAR'
hdr_gal['CRPIX2']   = 1440.5#720.5
hdr_gal['CRVAL2']   = 0.
hdr_gal['CDELT2']   = 0.0625#0.125
#print(repr(hdr_gal))

In [ ]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

## Calculate moments from original files

In [ ]:
drao_m = FD_moments(data_drao,0,400,FD_drao)
staps_m = FD_moments(data_staps,0,400,FD_staps)

#drao_m = FD_moments(data_drao,0.04,450,FD_drao)
#staps_m = FD_moments(data_staps,0.02,450,FD_staps)

## Extract M0, M1 and check maps:

In [ ]:
drao_m1_gal = drao_m['M1']
staps_m1_eq = staps_m['M1']
drao_m0_gal = drao_m['M0']
staps_m0_eq = staps_m['M0']

fig = plt.figure(figsize=(18,8))
ax1 = fig.add_subplot(221, projection=WCS(hdr_drao).celestial)
ax2 = fig.add_subplot(222, projection=WCS(hdr_drao).celestial)
ax3 = fig.add_subplot(223, projection=WCS(hdr_staps).celestial)
ax4 = fig.add_subplot(224, projection=WCS(hdr_staps).celestial)

ax1.imshow(drao_m0_gal,origin='lower',vmin=0,vmax=60,cmap='cubehelix')
ax2.imshow(drao_m1_gal,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')
ax3.imshow(staps_m0_eq,origin='lower',vmin=0,vmax=30,cmap='cubehelix')
ax4.imshow(staps_m1_eq,origin='lower',vmin=-50,vmax=50,cmap='RdBu_r')

## Make 2D versions of original headers

In [ ]:
hdr2D_drao = hdr_drao.copy()
del hdr2D_drao['NAXIS3']
del hdr2D_drao['CRPIX3']
del hdr2D_drao['CDELT3']
del hdr2D_drao['CUNIT3']
del hdr2D_drao['CTYPE3']
del hdr2D_drao['CRVAL3']
hdr2D_drao['NAXIS'] = 2
hdr2D_drao['WCSAXES'] = 2
#print(repr(hdr2D_drao))

hdr2D_staps = hdr_staps.copy()
del hdr2D_staps['NAXIS3']
del hdr2D_staps['CRPIX3']
del hdr2D_staps['CDELT3']
del hdr2D_staps['CTYPE3']
del hdr2D_staps['CRVAL3']
hdr2D_staps['NAXIS'] = 2
hdr2D_staps['WCSAXES'] = 2
#print(repr(hdr2D_staps))

## Make healpix maps of original M0, M1

In [ ]:
drao_m0_gal_hpx, footprint = reproject_to_healpix((drao_m0_gal, hdr2D_drao),'galactic', nside=1024)
staps_m0_eq_hpx, footprint = reproject_to_healpix((staps_m0_eq, hdr2D_staps),'ICRS', nside=1024)

drao_m1_gal_hpx, footprint = reproject_to_healpix((drao_m1_gal, hdr2D_drao),'galactic', nside=1024)
staps_m1_eq_hpx, footprint = reproject_to_healpix((staps_m1_eq, hdr2D_staps),'ICRS', nside=1024)

## Rotate DRAO to EQ and STAPS to GAL

In [ ]:
r1 = hp.Rotator(coord=['C', 'G'])
staps_m1_gal_hpx = np.array(r1.rotate_map_pixel(staps_m1_eq_hpx))
staps_m0_gal_hpx = np.array(r1.rotate_map_pixel(staps_m0_eq_hpx))

r2 = hp.Rotator(coord=['G', 'C'])
drao_m1_eq_hpx = np.array(r2.rotate_map_pixel(drao_m1_gal_hpx))
drao_m0_eq_hpx = np.array(r2.rotate_map_pixel(drao_m0_gal_hpx))

## Reproject from healpix to new, matched headers

In [ ]:
#staps_m0_gal, footprint = reproject_from_healpix((staps_m0_gal_hpx,'galactic'),hdr_gal, nested=False)
#staps_m1_gal, footprint = reproject_from_healpix((staps_m1_gal_hpx,'galactic'),hdr_gal, nested=False)

#drao_m0_gal,  footprint = reproject_from_healpix((drao_m0_gal_hpx, 'galactic'),hdr_gal, nested=False)
#drao_m1_gal,  footprint = reproject_from_healpix((drao_m1_gal_hpx, 'galactic'),hdr_gal, nested=False)

staps_m0_eq, footprint = reproject_from_healpix((staps_m0_eq_hpx,'ICRS'),hdr_eq, nested=False)
staps_m1_eq, footprint = reproject_from_healpix((staps_m1_eq_hpx,'ICRS'),hdr_eq, nested=False)

drao_m0_eq,  footprint = reproject_from_healpix((drao_m0_eq_hpx, 'ICRS'),hdr_eq, nested=False)
drao_m1_eq,  footprint = reproject_from_healpix((drao_m1_eq_hpx, 'ICRS'),hdr_eq, nested=False)

In [ ]:
print(drao_m1_eq.shape)
print(staps_m1_eq.shape)

## Check Equatorial maps:

In [ ]:
fig = plt.figure(figsize=(18,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_eq).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_eq).celestial)

crange = SkyCoord([359.9,0], [-30,90], frame=ICRS, unit=(u.deg, u.deg))
ax1.imshow(drao_m1_eq,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r')
ax1.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax1.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

crange = SkyCoord([359.9,0], [-90,30], frame=ICRS, unit=(u.deg, u.deg))
ax2.imshow(staps_m1_eq,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r')
ax2.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax2.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

In [ ]:
fig = plt.figure(figsize=(18,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_eq).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_eq).celestial)

crange = SkyCoord([359.9,0], [-30,90], frame=ICRS, unit=(u.deg, u.deg))
ax1.imshow(drao_m0_eq,origin='lower',vmin=0,vmax=60,cmap='cubehelix')
ax1.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax1.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

crange = SkyCoord([359.9,0], [-90,30], frame=ICRS, unit=(u.deg, u.deg))
ax2.imshow(staps_m0_eq,origin='lower',vmin=0,vmax=30,cmap='cubehelix')
ax2.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax2.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

## Convolve GMIMS-HBS M1 to GMIMS-HBN M1 beam

In [ ]:
drao_m1_eq[np.isnan(drao_m1_eq)] = 0.0
staps_m1_eq[np.isnan(staps_m1_eq)] = 0.0
drao_m0_eq[np.isnan(drao_m0_eq)] = 0.0
staps_m0_eq[np.isnan(staps_m0_eq)] = 0.0

In [ ]:
#staps_m1_eq_conv = convolve_to_beam(staps_m1_eq, 36/60, 18/60, hdr_eq, output = True)

In [ ]:
#print(staps_m1_eq_conv.shape)

In [ ]:
fig = plt.figure(figsize=(18,12))
ax1 = fig.add_subplot(211, projection=WCS(hdr_eq).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_eq).celestial)

crange = SkyCoord([359.9,0], [-30,90], frame=ICRS, unit=(u.deg, u.deg))
ax1.imshow(drao_m1_eq,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r')
ax1.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax1.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

crange = SkyCoord([359.9,0], [-90,30], frame=ICRS, unit=(u.deg, u.deg))
ax2.imshow(staps_m1_eq_conv,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r')
ax2.set_xlim(WCS(hdr_eq).world_to_pixel(crange)[0])
ax2.set_ylim(WCS(hdr_eq).world_to_pixel(crange)[1])

## Remove low-dec DRAO and check:

In [ ]:
#drao_m1_eq[0:713] = np.nan
drao_m1_eq[0:1427] = np.nan
for i in range(0,drao_m1_eq.shape[0]):
    print(i,drao_m1_eq[i,0],staps_m1_eq[i,0])

In [ ]:
#drao_m1_eq[0:713] = np.nan
drao_m0_eq[0:1427] = np.nan
for i in range(0,drao_m0_eq.shape[0]):
    print(i,drao_m0_eq[i,0],staps_m0_eq[i,0])

## Combine Equatorial STAPS and DRAO:

In [ ]:
drao_m1_eq[np.isnan(drao_m1_eq)] = 0.0
staps_m1_eq[np.isnan(staps_m1_eq)] = 0.0
drao_m0_eq[np.isnan(drao_m0_eq)] = 0.0
staps_m0_eq[np.isnan(staps_m0_eq)] = 0.0
#staps_m1_eq_conv[np.isnan(staps_m1_eq_conv)] = 0.0

#both_m1_eq = drao_m1_eq+staps_m1_eq_conv
both_m1_eq = drao_m1_eq+staps_m1_eq
both_m0_eq = drao_m0_eq+staps_m0_eq

In [ ]:
plt.imshow(both_m0_eq,vmin=0,vmax=50,origin='lower')

In [ ]:
both_m1_eq_hpx, footprint = reproject_to_healpix((both_m1_eq, hdr_eq),'ICRS', nside=512)
both_m0_eq_hpx, footprint = reproject_to_healpix((both_m0_eq, hdr_eq),'ICRS', nside=512)

r2 = hp.Rotator(coord=['C', 'G'])
both_m1_gal_hpx = np.array(r2.rotate_map_pixel(both_m1_eq_hpx))
both_m0_gal_hpx = np.array(r2.rotate_map_pixel(both_m0_eq_hpx))

both_m1_gal, footprint = reproject_from_healpix((both_m1_gal_hpx, 'galactic'),hdr_gal, nested=False)
both_m0_gal, footprint = reproject_from_healpix((both_m0_gal_hpx, 'galactic'),hdr_gal, nested=False)

## Plot combined

## Old method for sizing stars:

In [ ]:
fig = plt.figure(figsize=(20,14))
ax = fig.add_subplot(111, projection=WCS(hdr_eq).celestial)

ax.imshow(both_m1_eq,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r',interpolation='none')
star_coords = SkyCoord(ra=ra_constel*u.deg, dec=dec_constel*u.deg,frame=ICRS)

print(len(star_coords))
for i in range(0,len(star_coords)):
    if np.isfinite(np.array(dec_constel)[i]):
        circle = mpath.Path.unit_circle()
        verts = np.copy(circle.vertices)
        stretch = np.round(1/abs(np.cos(np.array(dec_constel)[i]*np.pi/180)),2)
        verts[:, 0] *= stretch
        ellipt_marker = mpath.Path(verts, circle.codes)

        ax.scatter([WCS(hdr_eq).world_to_pixel(star_coords)[0][i]],
                    [WCS(hdr_eq).world_to_pixel(star_coords)[1][i]],color='k',
                    s=0.8*np.array(marker_size)[i],marker=ellipt_marker)

ax.tick_params(axis='both', which='major', labelsize=14)

tick_positionsx = np.arange(0, 360, 30)
tick_positionsy = np.arange(-90, 120, 30)
ax.coords[0].set_ticks(tick_positionsx * u.deg)  
ax.coords[1].set_ticks(tick_positionsy * u.deg)  
ax.coords[0].set_major_formatter('d')
ax.coords[1].set_major_formatter('d')
ax.coords[0].display_minor_ticks(True)
ax.coords[1].display_minor_ticks(True)
ax.set_xlabel('longitude',fontsize=14)
ax.set_ylabel('latitude',fontsize=14)

ax.grid()
plt.savefig('../sky_map_with_grid_old_check.pdf')

## New method for sizing stars

In [ ]:


fig = plt.figure(figsize=(20,14))
ax = fig.add_subplot(111, projection=WCS(hdr_eq).celestial)

ax.imshow(both_m1_eq,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r',interpolation='none')
star_coords = SkyCoord(ra=ra_constel*u.deg, dec=dec_constel*u.deg,frame=ICRS)

ells = [Ellipse(xy = (WCS(hdr_eq).world_to_pixel(star_coords)[0][i],
                      WCS(hdr_eq).world_to_pixel(star_coords)[1][i]), 
                width  = 4*np.sqrt(marker_size[i])/np.cos(dec_constel[i]*np.pi/180), 
                height = 4*np.sqrt(marker_size[i])) 
        for i in range(0,len(ra_constel))]

# Uncomment for making circular shaped stars:
#ells = [Ellipse(xy = (WCS(hdr_eq).world_to_pixel(star_coords)[0][i],
#                      WCS(hdr_eq).world_to_pixel(star_coords)[1][i]), 
#                width  = 4*np.sqrt(marker_size[i]), 
#                height = 4*np.sqrt(marker_size[i])) 
#        for i in range(0,len(ra_constel))]

for e in ells:
    #print(e)
    ax.add_artist(e)
    e.set_clip_box(ax.bbox)
    e.set_facecolor('k')

ax.tick_params(axis='both', which='major', labelsize=14)

tick_positionsx = np.arange(0, 360, 30)
tick_positionsy = np.arange(-90, 120, 30)
ax.coords[0].set_ticks(tick_positionsx * u.deg)  
ax.coords[1].set_ticks(tick_positionsy * u.deg)  
ax.coords[0].set_major_formatter('d')
ax.coords[1].set_major_formatter('d')
ax.coords[0].display_minor_ticks(True)
ax.coords[1].display_minor_ticks(True)
ax.set_xlabel('longitude',fontsize=14)
ax.set_ylabel('latitude',fontsize=14)

ax.grid()
#plt.savefig('../sky_map_with_grid_new_circles.pdf')
plt.savefig('../sky_map_with_grid_Apr03.eps')

In [ ]:
np.pi*16/12

In [ ]:
both_m1_gal_hpx[both_m1_gal_hpx==0]=np.nan

fig = plt.figure()
hp.orthview(both_m1_gal_hpx,min=-50,max=50,cmap='RdBu_r',rot=(0,0),xsize=2000,badcolor='k')
hp.graticule()
plt.title('Galactic', fontsize=14)
#plt.savefig('../gmims_map_eq_orth.pdf', dpi=1200, bbox_inches='tight')

In [ ]:
both_m1_eq_hpx[both_m1_eq_hpx==0]=np.nan

fig = plt.figure()
hp.orthview(both_m1_eq_hpx,min=-80,max=80,cmap='RdBu_r',rot=(0,90),xsize=2000,badcolor='k')
hp.graticule()
plt.title('Equatorial', fontsize=14)
#plt.savefig('../gmims_map_eq_orth.pdf', dpi=1200, bbox_inches='tight')

## Old code

## Check Galactic maps:

In [ ]:
fig = plt.figure(figsize=(18,18))
ax1 = fig.add_subplot(211, projection=WCS(hdr_gal).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_gal).celestial)

crange = SkyCoord([359.9,0], [-90,90], frame=Galactic, unit=(u.deg, u.deg))
ax1.imshow(drao_m1_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')
ax1.set_xlim(WCS(hdr_gal).world_to_pixel(crange)[0])
ax1.set_ylim(WCS(hdr_gal).world_to_pixel(crange)[1])

crange = SkyCoord([359.9,0], [-90,90], frame=Galactic, unit=(u.deg, u.deg))
ax2.imshow(staps_m1_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r')
ax2.set_xlim(WCS(hdr_gal).world_to_pixel(crange)[0])
ax2.set_ylim(WCS(hdr_gal).world_to_pixel(crange)[1])

In [ ]:
fig = plt.figure(figsize=(18,18))
ax1 = fig.add_subplot(211, projection=WCS(hdr_gal).celestial)
ax2 = fig.add_subplot(212, projection=WCS(hdr_gal).celestial)

crange = SkyCoord([359.9,0], [-90,90], frame=Galactic, unit=(u.deg, u.deg))
ax1.imshow(drao_m0_gal,origin='lower',vmin=0,vmax=60,cmap='cubehelix')
ax1.set_xlim(WCS(hdr_gal).world_to_pixel(crange)[0])
ax1.set_ylim(WCS(hdr_gal).world_to_pixel(crange)[1])

crange = SkyCoord([359.9,0], [-90,90], frame=Galactic, unit=(u.deg, u.deg))
ax2.imshow(staps_m0_gal,origin='lower',vmin=0,vmax=30,cmap='cubehelix')
ax2.set_xlim(WCS(hdr_gal).world_to_pixel(crange)[0])
ax2.set_ylim(WCS(hdr_gal).world_to_pixel(crange)[1])

In [ ]:
from astroquery.simbad import Simbad

def get_major_star_coordinates(constellation_name):
    simbad_query = Simbad.query_region(constellation_name, radius=1.0*u.deg)
    star_info = simbad_query.to_pandas()

    stars_info = []
    for _, row in star_info.iterrows():
        star_name = row['MAIN_ID'].decode("utf-8") if isinstance(row['MAIN_ID'], bytes) else row['MAIN_ID']
        ra = row['RA']
        dec = row['DEC']
        stars_info.append((star_name, ra, dec))

    return stars_info

def generate_constellations(constellation_names):
    constellations_dict = {}

    for constellation_name in constellation_names:
        stars_info = get_major_star_coordinates(constellation_name)
        constellations_dict[constellation_name] = stars_info

    return constellations_dict

# Example usage
#constellation_names = ['Orion', 'Ursa Major', 'Cygnus']
constellation_names = ['Orion']
constellations = generate_constellations(constellation_names)
print(constellations)

In [ ]:
fig = plt.figure(figsize=(20,16))
ax = fig.add_subplot(111, projection=WCS(hdr_gal).celestial)

ax.imshow(both_m1_gal,origin='lower',vmin=-80,vmax=80,cmap='RdBu_r',interpolation='none')
ax.grid()
plt.savefig('../gmims_map_gal.pdf')

In [ ]:
both_m1_gal_hpx[both_m1_gal_hpx==0]=np.nan

In [ ]:
fig = plt.figure()
hp.orthview(both_m1_gal_hpx,min=-80,max=80,cmap='RdBu_r',rot=(123,0),xsize=2000,badcolor='k')
hp.graticule()
plt.title('Galactic', fontsize=14)
plt.savefig('../gmims_map_gal_orth.pdf', dpi=1200, bbox_inches='tight')


## Make Mollweide plot for Alex's Stanford slides

In [ ]:
fs = 14

hp.newvisufunc.projview(both_m0_gal_hpx,cmap='cubehelix',min=0,max=50,rot=(0,0),
                            xsize=2000,cbar=True,title='GMIMS-HBN and STAPS M0',unit=r'K RMSF$^{-1}$ rad m$^{-2}$',
                            projection_type='mollweide',fontsize={'cbar_label':fs,'cbar_tick_label':fs,'title':fs+2},
                            override_plot_properties={'cbar_shrink':0.4})
plt.savefig('/home/aordog/GMIMS_PLOTS/all_gmims_May2024/HBN_and_HBS_M0_PIthresh.png',dpi=200,transparent=True)

In [ ]:
import copy
both_m1_gal_hpx_new = copy.deepcopy(both_m1_gal_hpx)
both_m1_gal_hpx_new[both_m1_gal_hpx_new == 0] = np.nan

In [ ]:
fs = 14

hp.newvisufunc.projview(both_m1_gal_hpx_new,cmap='RdBu_r',min=-50,max=50,rot=(0,0),
                            xsize=2000,cbar=True,title='GMIMS-HBN and STAPS M1',unit=r'rad m$^{-2}$',
                            projection_type='mollweide',fontsize={'cbar_label':fs,'cbar_tick_label':fs,'title':fs+2},
                            override_plot_properties={'cbar_shrink':0.4})
plt.savefig('/home/aordog/GMIMS_PLOTS/all_gmims_May2024/HBN_and_HBS_M1_PIthresh.png',dpi=200,transparent=True)

In [ ]:
coords = SkyCoord(l=const_lon_all*u.deg, b=const_lat_all*u.deg,frame=Galactic)
l_constel  = coords.galactic.l.deg
b_constel = coords.galactic.b.deg

fig = plt.figure(figsize=(20,14))
ax = fig.add_subplot(111, projection=WCS(hdr_gal).celestial)

ax.imshow(both_m1_gal,origin='lower',vmin=-60,vmax=60,cmap='RdBu_r',interpolation='none')
star_coords = SkyCoord(l=l_constel*u.deg, b=b_constel*u.deg,frame=Galactic)



#ells = [Ellipse(xy = (WCS(hdr_eq).world_to_pixel(star_coords)[0][i],
#                      WCS(hdr_eq).world_to_pixel(star_coords)[1][i]), 
#                width  = 4*np.sqrt(marker_size[i])/np.cos(dec_constel[i]*np.pi/180), 
#                height = 4*np.sqrt(marker_size[i])) 
#        for i in range(0,len(ra_constel))]

# Uncomment for making circular shaped stars:
ells = [Ellipse(xy = (WCS(hdr_gal).world_to_pixel(star_coords)[0][i],
                      WCS(hdr_gal).world_to_pixel(star_coords)[1][i]), 
                width  = 4*np.sqrt(marker_size[i]), 
                height = 4*np.sqrt(marker_size[i])) 
        for i in range(0,len(l_constel))]

for e in ells:
    #print(e)
    ax.add_artist(e)
    e.set_clip_box(ax.bbox)
    e.set_facecolor('k')

ax.tick_params(axis='both', which='major', labelsize=14)

tick_positionsx = np.arange(-180, 180, 30)
tick_positionsy = np.arange(-90, 120, 30)
ax.coords[0].set_ticks(tick_positionsx * u.deg)  
ax.coords[1].set_ticks(tick_positionsy * u.deg)  
ax.coords[0].set_major_formatter('d')
ax.coords[1].set_major_formatter('d')
ax.coords[0].display_minor_ticks(True)
ax.coords[1].display_minor_ticks(True)
ax.set_xlabel('longitude',fontsize=14)
ax.set_ylabel('latitude',fontsize=14)

ax.grid()
#plt.savefig('../sky_map_with_grid_new_circles.pdf')
plt.savefig('../sky_map_with_gal_grid_open_house_slides.png')